# Gemma 4 OCR Fine-Tuning Colab v2

Full Colab training notebook for `google/gemma-4-12b-it` OCR/document extraction fine-tuning with LoRA, Google Drive dataset loading, Hugging Face checkpoint sync, T4 low-VRAM settings, and safe resume recovery for broken Adam optimizer state (`KeyError: 'exp_avg'`).

Expected Drive dataset archive: `docmind-ocr-dataset.zip` containing `ocr-images-sft.jsonl` plus image files.

In [ ]:
# Cell 1 - Install dependencies
!pip -q install -U "transformers>=4.56.0" "accelerate>=1.1.0" "peft>=0.13.2" "datasets>=3.0.0" "huggingface_hub>=0.26.0" "safetensors>=0.4.5" pillow tqdm

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# Cell 2 - Imports, Drive, Hugging Face, and configuration
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
import gc
import json
import math
import os
import random
import shutil
import zipfile
from typing import Any, Dict, List, Optional, Tuple, Union

import torch
from PIL import Image
from tqdm.auto import tqdm

from google.colab import drive
from huggingface_hub import HfApi, hf_hub_download, login, snapshot_download
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import Dataset
from transformers import AutoModelForImageTextToText, AutoProcessor, Trainer, TrainingArguments, set_seed


def log(message: str) -> None:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")


@dataclass
class TrainConfig:
    model_id: str = "google/gemma-4-12b-it"
    hf_checkpoint_repo: str = "moaziz2030/docmind-gemma4-ckpt"
    drive_root: str = "/content/drive/MyDrive/docmind_gemma4"
    dataset_zip_name: str = "docmind-ocr-dataset.zip"
    extracted_dataset_dir: str = "/content/dataset"
    data_jsonl_name: str = "ocr-images-sft.jsonl"
    checkpoint_dir_name: str = "checkpoints"
    output_dir: str = "/content/gemma4-docmind-output"
    seed: int = 42
    sequence_length: int = 512
    image_size: int = 336
    lora_rank: int = 4
    lora_alpha: int = 8
    lora_dropout: float = 0.05
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 16
    learning_rate: float = 2e-4
    weight_decay: float = 0.01
    num_train_epochs: float = 1.0
    logging_steps: int = 1
    eval_steps: int = 50
    save_steps: int = 50
    save_total_limit: int = 3
    warmup_ratio: float = 0.03
    validation_split: float = 0.10
    all_data: bool = True
    low_vram: bool = True
    bf16: bool = True
    fp16: bool = False
    auto_push_to_hub: bool = True


CFG = TrainConfig()
set_seed(CFG.seed)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"GPU cleaned: {free_bytes / 1024**3:.2f} GB free / {total_bytes / 1024**3:.2f} GB total")
    print(f"Device: cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {total_bytes / 1024**3:.1f} GB")
else:
    print("Device: cpu")

DTYPE = torch.bfloat16 if CFG.bf16 and torch.cuda.is_available() else torch.float16 if CFG.fp16 and torch.cuda.is_available() else torch.float32
print(f"Settings: seq={CFG.sequence_length}  r={CFG.lora_rank}  grad_accum={CFG.gradient_accumulation_steps}  img={CFG.image_size}  low_vram={CFG.low_vram}  dtype={DTYPE}")

drive.mount("/content/drive")
DRIVE_ROOT = Path(CFG.drive_root)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = DRIVE_ROOT / CFG.checkpoint_dir_name
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoint Dir: {CHECKPOINT_DIR}")

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    login()

api = HfApi()
api.create_repo(CFG.hf_checkpoint_repo, repo_type="model", private=True, exist_ok=True)
print(f"✅ HF repo ready: {CFG.hf_checkpoint_repo}")

In [ ]:
# Cell 3 - Extract and load OCR dataset
log("Loading dataset from Google Drive...")

zip_path = DRIVE_ROOT / CFG.dataset_zip_name
if not zip_path.exists():
    matches = list(DRIVE_ROOT.glob("*.zip"))
    if len(matches) == 1:
        zip_path = matches[0]
    else:
        raise FileNotFoundError(f"Could not find {CFG.dataset_zip_name} under {DRIVE_ROOT}")

extract_dir = Path(CFG.extracted_dataset_dir)
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True, exist_ok=True)

print(f"✅ Found: {zip_path.name} ({zip_path.stat().st_size / 1024**2:.0f} MB) — extracting...")
with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_dir)
print("✅ Dataset extracted")

jsonl_candidates = list(extract_dir.rglob(CFG.data_jsonl_name))
if not jsonl_candidates:
    jsonl_candidates = list(extract_dir.rglob("*.jsonl"))
if not jsonl_candidates:
    raise FileNotFoundError(f"No JSONL dataset found under {extract_dir}")

DATA_JSONL = jsonl_candidates[0]
INPUT_DIR = DATA_JSONL.parent
print(f"INPUT_DIR : {INPUT_DIR}")
print(f"DATA_JSONL: {DATA_JSONL}")


def normalize_record(raw: Dict[str, Any], base_dir: Path) -> Optional[Dict[str, str]]:
    image_value = raw.get("image") or raw.get("image_path") or raw.get("file") or raw.get("filename") or raw.get("path")
    answer_value = raw.get("text") or raw.get("target") or raw.get("answer") or raw.get("output") or raw.get("ocr") or raw.get("markdown")
    prompt_value = raw.get("prompt") or raw.get("instruction") or "Extract all readable text from this document image. Preserve line breaks and table-like structure when useful."

    if not image_value or answer_value is None:
        return None

    image_path = Path(str(image_value))
    if not image_path.is_absolute():
        image_path = base_dir / image_path
    if not image_path.exists():
        alt_matches = list(base_dir.rglob(Path(str(image_value)).name))
        if alt_matches:
            image_path = alt_matches[0]
        else:
            return None

    answer = str(answer_value).strip()
    if not answer:
        return None

    return {
        "image_path": str(image_path),
        "prompt": str(prompt_value).strip(),
        "answer": answer,
    }


records: List[Dict[str, str]] = []
skipped = 0
print(f"\nLoading records (all_data={CFG.all_data})...")
with DATA_JSONL.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue
        try:
            raw_record = json.loads(line)
        except json.JSONDecodeError:
            skipped += 1
            continue
        record = normalize_record(raw_record, INPUT_DIR)
        if record is None:
            skipped += 1
            continue
        records.append(record)

random.Random(CFG.seed).shuffle(records)
val_count = max(1, int(len(records) * CFG.validation_split))
val_records = records[:val_count]
train_records = records[val_count:]
print(f"Loaded {len(records)} records | {skipped} skipped (image not found or invalid)")
print(f"Train: {len(train_records)}  Val: {len(val_records)}")

In [ ]:
# Cell 4 - Dataset and collatorfrom typing import Any, Dict, Listimport torchfrom PIL import Imagefrom torch.utils.data import Datasetfrom transformers import AutoProcessorclass OCRImageDataset(Dataset):
    def __init__(self, items: List[Dict[str, str]]):
        self.items = items

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, index: int) -> Dict[str, str]:
        return self.items[index]


def open_rgb_image(path: str) -> Image.Image:
    image = Image.open(path).convert("RGB")
    image.thumbnail((CFG.image_size, CFG.image_size), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (CFG.image_size, CFG.image_size), "white")
    left = (CFG.image_size - image.width) // 2
    top = (CFG.image_size - image.height) // 2
    canvas.paste(image, (left, top))
    return canvas


def build_messages(example: Dict[str, str], include_answer: bool = True) -> List[Dict[str, Any]]:
    messages: List[Dict[str, Any]] = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": example["prompt"]},
            ],
        }
    ]
    if include_answer:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": example["answer"]}]})
    return messages


class OCRDataCollator:
    def __init__(self, processor: AutoProcessor):
        self.processor = processor

    def __call__(self, examples: List[Dict[str, str]]) -> Dict[str, torch.Tensor]:
        images = [open_rgb_image(example["image_path"]) for example in examples]
        texts = [
            self.processor.apply_chat_template(
                build_messages(example, include_answer=True),
                tokenize=False,
                add_generation_prompt=False,
            )
            for example in examples
        ]
        user_prefixes = [
            self.processor.apply_chat_template(
                build_messages(example, include_answer=False),
                tokenize=False,
                add_generation_prompt=True,
            )
            for example in examples
        ]

        batch = self.processor(
            text=texts,
            images=images,
            padding=True,
            truncation=True,
            max_length=CFG.sequence_length,
            return_tensors="pt",
        )
        labels = batch["input_ids"].clone()

        for row_index, user_prefix in enumerate(user_prefixes):
            prefix_ids = self.processor.tokenizer(
                user_prefix,
                add_special_tokens=False,
                truncation=True,
                max_length=CFG.sequence_length,
            )["input_ids"]
            prefix_length = min(len(prefix_ids), labels.shape[1])
            labels[row_index, :prefix_length] = -100

        if self.processor.tokenizer.pad_token_id is not None:
            labels[batch["input_ids"] == self.processor.tokenizer.pad_token_id] = -100

        batch["labels"] = labels
        return batch


train_dataset = OCRImageDataset(train_records)
val_dataset = OCRImageDataset(val_records)
print(f"✅ Dataset ready — train: {len(train_dataset)}  val: {len(val_dataset)}")

In [ ]:
# Cell 5 - Load processor/model and apply LoRA
if CFG.low_vram:
    print("ℹ️  caching_allocator_warmup disabled (T4 mode)")

log(f"Loading {CFG.model_id}...")
processor = AutoProcessor.from_pretrained(CFG.model_id, trust_remote_code=True)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    CFG.model_id,
    dtype=DTYPE,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.config.use_cache = False

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

try:
    model = prepare_model_for_kbit_training(model)
except Exception as exc:
    print(f"prepare_model_for_kbit_training skipped: {type(exc).__name__}: {exc}")

lora_config = LoraConfig(
    r=CFG.lora_rank,
    lora_alpha=CFG.lora_alpha,
    lora_dropout=CFG.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)


def install_embed_vision_float_guard(model_obj: torch.nn.Module) -> bool:
    installed = False

    def cast_first_tensor_to_module_dtype(module, inputs):
        if not inputs:
            return inputs
        first = inputs[0]
        try:
            module_dtype = next(module.parameters()).dtype
        except StopIteration:
            return inputs
        if torch.is_tensor(first) and first.is_floating_point() and first.dtype != module_dtype:
            return (first.to(module_dtype), *inputs[1:])
        return inputs

    for module_name, module in model_obj.named_modules():
        lowered = module_name.lower()
        if "vision" in lowered and ("embed" in lowered or "patch" in lowered):
            module.register_forward_pre_hook(cast_first_tensor_to_module_dtype)
            installed = True

    return installed

if install_embed_vision_float_guard(model):
    print("✅ embed_vision float-guard hook installed")
else:
    print("ℹ️ embed_vision float-guard hook not needed or not found")

model.print_trainable_parameters()
log("✅ Model loaded and LoRA applied")

In [ ]:
# Cell 6 - Safe checkpoint discovery and resume helpers
CheckpointLike = Union[str, Path, None]
TRAINER_STATE_FILES_TO_STRIP = ("optimizer.pt", "scheduler.pt", "trainer_state.json", "rng_state.pth", "scaler.pt")
ADAM_REQUIRED_STATE_KEYS = ("exp_avg", "exp_avg_sq")


def checkpoint_step(path: Path) -> int:
    try:
        return int(path.name.split("checkpoint-")[-1])
    except Exception:
        return -1


def lora_target_modules_from_checkpoint(checkpoint_dir: Path) -> List[str]:
    adapter_config_path = checkpoint_dir / "adapter_config.json"
    if not adapter_config_path.exists():
        return []
    try:
        config = json.loads(adapter_config_path.read_text(encoding="utf-8"))
    except Exception:
        return []
    target_modules = config.get("target_modules") or []
    return sorted(str(module) for module in target_modules)


def current_lora_target_modules() -> List[str]:
    return sorted(str(module) for module in lora_config.target_modules)


def optimizer_state_is_adam_safe(checkpoint_dir: CheckpointLike) -> Tuple[bool, str]:
    if checkpoint_dir is None:
        return False, "no checkpoint selected"

    checkpoint_path = Path(checkpoint_dir)
    optimizer_path = checkpoint_path / "optimizer.pt"

    if not checkpoint_path.exists():
        return False, f"checkpoint does not exist: {checkpoint_path}"
    if not optimizer_path.exists():
        return False, "optimizer.pt missing"

    try:
        optimizer_payload = torch.load(optimizer_path, map_location="cpu")
    except Exception as exc:
        return False, f"optimizer.pt unreadable: {type(exc).__name__}: {exc}"

    if not isinstance(optimizer_payload, dict):
        return False, "optimizer.pt payload is not a dictionary"

    optimizer_state = optimizer_payload.get("state")
    parameter_groups = optimizer_payload.get("param_groups")

    if not isinstance(optimizer_state, dict) or not optimizer_state:
        return False, "optimizer state is empty"
    if not isinstance(parameter_groups, list) or not parameter_groups:
        return False, "optimizer parameter groups are empty"

    broken_parameter_ids = []
    for parameter_id, parameter_state in optimizer_state.items():
        if not isinstance(parameter_state, dict):
            broken_parameter_ids.append(parameter_id)
            continue
        for required_key in ADAM_REQUIRED_STATE_KEYS:
            value = parameter_state.get(required_key)
            if not torch.is_tensor(value):
                broken_parameter_ids.append(parameter_id)
                break

    if broken_parameter_ids:
        return False, f"Adam slots missing or invalid for {len(broken_parameter_ids)} parameter states"

    return True, "optimizer state looks compatible"


def strip_trainer_resume_state(checkpoint_dir: CheckpointLike) -> Path:
    if checkpoint_dir is None:
        raise ValueError("checkpoint_dir is required")

    checkpoint_path = Path(checkpoint_dir)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    stripped_path = checkpoint_path.parent / f"{checkpoint_path.name}-weights-only"
    if stripped_path.exists():
        shutil.rmtree(stripped_path)
    shutil.copytree(checkpoint_path, stripped_path)

    for filename in TRAINER_STATE_FILES_TO_STRIP:
        path = stripped_path / filename
        if path.exists():
            path.unlink()
    for rng_path in stripped_path.glob("rng_state_*.pth"):
        rng_path.unlink()

    return stripped_path


def checkpoint_is_lora_compatible(checkpoint_dir: Path) -> Tuple[bool, str]:
    checkpoint_targets = lora_target_modules_from_checkpoint(checkpoint_dir)
    current_targets = current_lora_target_modules()
    if checkpoint_targets and checkpoint_targets != current_targets:
        return False, f"Checkpoint modules {checkpoint_targets} ≠ current {current_targets}"
    return True, "LoRA modules compatible"


def download_latest_hf_checkpoint() -> Optional[Path]:
    log(f"   Checking HF Hub: {CFG.hf_checkpoint_repo}")
    try:
        repo_files = api.list_repo_files(CFG.hf_checkpoint_repo, repo_type="model")
    except Exception as exc:
        print(f"   HF Hub checkpoint scan skipped: {type(exc).__name__}: {exc}")
        return None

    checkpoint_names = sorted({Path(file).parts[0] for file in repo_files if file.startswith("checkpoint-")}, key=lambda name: int(name.split("checkpoint-")[-1]), reverse=True)
    for checkpoint_name in checkpoint_names:
        local_path = CHECKPOINT_DIR / checkpoint_name
        print(f"   Downloading {checkpoint_name} from HF Hub...")
        try:
            snapshot_download(
                repo_id=CFG.hf_checkpoint_repo,
                repo_type="model",
                allow_patterns=f"{checkpoint_name}/**",
                local_dir=str(CHECKPOINT_DIR),
                local_dir_use_symlinks=False,
            )
        except Exception as exc:
            print(f"   Could not download {checkpoint_name}: {type(exc).__name__}: {exc}")
            continue
        if local_path.exists():
            return local_path
    return None


def find_resume_checkpoint() -> Optional[Path]:
    log("🔍 Searching for checkpoint to resume from...")
    local_checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"), key=checkpoint_step, reverse=True)

    for checkpoint_dir in local_checkpoints:
        if checkpoint_dir.name.endswith("-weights-only"):
            continue
        compatible, reason = checkpoint_is_lora_compatible(checkpoint_dir)
        if compatible:
            print(f"   ✅ Local checkpoint compatible: {checkpoint_dir.name}")
            return checkpoint_dir
        print(f"   ⚠️  {reason} — skipping")
        print(f"   Skipping incompatible: {checkpoint_dir.name}")

    hf_checkpoint = download_latest_hf_checkpoint()
    if hf_checkpoint:
        compatible, reason = checkpoint_is_lora_compatible(hf_checkpoint)
        if compatible:
            return hf_checkpoint
        print(f"   ⚠️  {reason} — HF checkpoint skipped")

    return None


def resolve_safe_resume_checkpoint(resume_from: CheckpointLike) -> Optional[str]:
    if resume_from is None:
        print("Resume: no compatible checkpoint selected; starting fresh.")
        return None

    resume_path = Path(resume_from)
    optimizer_ok, optimizer_reason = optimizer_state_is_adam_safe(resume_path)
    print(f"Resume optimizer check: {optimizer_reason}")

    if optimizer_ok:
        print(f"✅ Full trainer resume enabled: {resume_path}")
        return str(resume_path)

    weights_only_path = strip_trainer_resume_state(resume_path)
    print(f"⚠️ Optimizer resume disabled. Loading weights only from: {weights_only_path}")
    print("   A fresh optimizer/scheduler will be created for the current LoRA target modules.")
    return str(weights_only_path)


def train_with_safe_resume(trainer: Trainer, resume_from: CheckpointLike = None):
    safe_resume_checkpoint = resolve_safe_resume_checkpoint(resume_from)
    try:
        return trainer.train(resume_from_checkpoint=safe_resume_checkpoint)
    except KeyError as exc:
        missing_key = str(exc).strip("'\"")
        if missing_key in ADAM_REQUIRED_STATE_KEYS and resume_from is not None:
            print(f"⚠️ Adam optimizer state is incompatible ({exc}). Retrying weights-only resume.")
            weights_only_path = strip_trainer_resume_state(resume_from)
            return trainer.train(resume_from_checkpoint=str(weights_only_path))
        raise
    except ValueError as exc:
        if "optimizer" in str(exc).lower() and resume_from is not None:
            print(f"⚠️ Optimizer state is incompatible ({exc}). Retrying weights-only resume.")
            weights_only_path = strip_trainer_resume_state(resume_from)
            return trainer.train(resume_from_checkpoint=str(weights_only_path))
        raise


def compute_warmup_steps(train_dataset_length: int, gradient_accumulation_steps: int, num_train_epochs: Union[int, float], warmup_ratio: float) -> int:
    safe_gradient_accumulation = max(1, int(gradient_accumulation_steps))
    safe_epochs = max(1, int(math.ceil(float(num_train_epochs))))
    updates_per_epoch = max(1, train_dataset_length // safe_gradient_accumulation)
    estimated_update_steps = max(1, updates_per_epoch * safe_epochs)
    return max(1, int(estimated_update_steps * warmup_ratio))

resume_from = find_resume_checkpoint()
if resume_from:
    print(f"✅ Candidate resume checkpoint: {resume_from}")
else:
    print("No compatible checkpoint found; training will start fresh.")

In [ ]:
# Cell 7 - Trainer setup and training
warmup_steps = compute_warmup_steps(
    train_dataset_length=len(train_dataset),
    gradient_accumulation_steps=CFG.gradient_accumulation_steps,
    num_train_epochs=CFG.num_train_epochs,
    warmup_ratio=CFG.warmup_ratio,
)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    overwrite_output_dir=False,
    per_device_train_batch_size=CFG.per_device_train_batch_size,
    per_device_eval_batch_size=CFG.per_device_eval_batch_size,
    gradient_accumulation_steps=CFG.gradient_accumulation_steps,
    learning_rate=CFG.learning_rate,
    weight_decay=CFG.weight_decay,
    num_train_epochs=CFG.num_train_epochs,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    logging_steps=CFG.logging_steps,
    eval_strategy="steps",
    eval_steps=CFG.eval_steps,
    save_strategy="steps",
    save_steps=CFG.save_steps,
    save_total_limit=CFG.save_total_limit,
    bf16=CFG.bf16,
    fp16=CFG.fp16,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    report_to="none",
    push_to_hub=False,
    gradient_checkpointing=True,
    optim="adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=OCRDataCollator(processor),
)

log("\n" + "=" * 60)
log("STARTING TRAINING")
log("=" * 60)

train_result = train_with_safe_resume(trainer, resume_from)
trainer.save_model(str(CHECKPOINT_DIR / "final-adapter"))
processor.save_pretrained(str(CHECKPOINT_DIR / "final-adapter"))
print(train_result)
log("✅ Training complete")

In [ ]:
# Cell 8 - Push latest checkpoints/adapters to Hugging Face Hub
if CFG.auto_push_to_hub:
    log(f"✅ Auto-push enabled: {CFG.hf_checkpoint_repo}")
    api.upload_folder(
        repo_id=CFG.hf_checkpoint_repo,
        repo_type="model",
        folder_path=str(CHECKPOINT_DIR),
        path_in_repo=".",
        ignore_patterns=["*-weights-only/**", "**/optimizer.pt", "**/scheduler.pt", "**/rng_state*.pth", "**/scaler.pt"],
        commit_message="Upload Gemma 4 OCR LoRA checkpoints from Colab",
    )
    log("✅ Checkpoints pushed to HF Hub")
else:
    log("Auto-push disabled")